In [2]:
import pandas as pd
import numpy as np
import requests
import time
import joblib

from datetime import datetime

from google_play_scraper import reviews, Sort

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report

In [3]:
DISNEY_APP_ID = "com.disney.disneyplus"
MAX_APP_ID = "com.wbd.stream"

In [4]:
def pull_reviews_large(app_id, target=10000):

    all_reviews = []
    token = None

    while len(all_reviews) < target:

        batch, token = reviews(
            app_id,
            lang="en",
            country="us",
            sort=Sort.NEWEST,
            count=200,
            continuation_token=token
        )

        if not batch:
            break

        all_reviews.extend(batch)

        print("Collected:", len(all_reviews))

        if token is None:
            break

        time.sleep(1)

    return all_reviews[:target]

In [5]:
print("Pulling Disney+ reviews...")
disney_big = pull_reviews_large(DISNEY_APP_ID, 10000)

print("Pulling Max reviews...")
max_big = pull_reviews_large(MAX_APP_ID, 10000)

print("Disney:", len(disney_big))
print("Max:", len(max_big))

Pulling Disney+ reviews...
Collected: 200
Collected: 400
Collected: 600
Collected: 800
Collected: 1000
Collected: 1200
Collected: 1400
Collected: 1600
Collected: 1800
Collected: 2000
Collected: 2200
Collected: 2400
Collected: 2600
Collected: 2800
Collected: 3000
Collected: 3200
Collected: 3400
Collected: 3600
Collected: 3800
Collected: 4000
Collected: 4200
Collected: 4400
Collected: 4600
Collected: 4800
Collected: 5000
Collected: 5200
Collected: 5400
Collected: 5600
Collected: 5800
Collected: 6000
Collected: 6200
Collected: 6400
Collected: 6600
Collected: 6800
Collected: 7000
Collected: 7200
Collected: 7400
Collected: 7600
Collected: 7800
Collected: 8000
Collected: 8200
Collected: 8400
Collected: 8600
Collected: 8800
Collected: 9000
Collected: 9200
Collected: 9400
Collected: 9600
Collected: 9800
Collected: 10000
Pulling Max reviews...
Collected: 200
Collected: 400
Collected: 600
Collected: 800
Collected: 1000
Collected: 1200
Collected: 1400
Collected: 1600
Collected: 1800
Collected: 20

In [6]:
def build_training_df(play_reviews, partner_name):

    rows = []

    for r in play_reviews:

        score = r["score"]

        if score <= 2:
            label = "Negative"
        elif score == 3:
            label = "Neutral"
        else:
            label = "Positive"

        rows.append({
            "partner": partner_name,
            "review_id": r["reviewId"],
            "text": r["content"],
            "score": score,
            "label": label
        })

    return pd.DataFrame(rows)

In [7]:
train_disney = build_training_df(disney_big, "Disney+")
train_max = build_training_df(max_big, "Max")

train_df = pd.concat([train_disney, train_max], ignore_index=True)

print("Total rows:", len(train_df))
print(train_df["label"].value_counts())

Total rows: 20000
label
Negative    10099
Positive     8470
Neutral      1431
Name: count, dtype: int64


In [8]:
train_df["text"] = train_df["text"].fillna("")
train_df = train_df[train_df["text"].str.strip() != ""]
train_df["text"] = train_df["text"].astype(str)

In [9]:
X = train_df["text"]
y = train_df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

model = Pipeline([
    ("tfidf", TfidfVectorizer(
        stop_words="english",
        ngram_range=(1,2),
        max_features=20000
    )),
    ("rf", RandomForestClassifier(
        n_estimators=300,
        n_jobs=-1,
        random_state=42
    ))
])

model.fit(X_train, y_train)

preds = model.predict(X_test)

print(classification_report(y_test, preds))

              precision    recall  f1-score   support

    Negative       0.82      0.91      0.86      2020
     Neutral       0.33      0.01      0.01       286
    Positive       0.85      0.88      0.87      1694

    accuracy                           0.83      4000
   macro avg       0.67      0.60      0.58      4000
weighted avg       0.80      0.83      0.80      4000



In [10]:
joblib.dump(model, "sentiment_rf_model.pkl")

['sentiment_rf_model.pkl']

In [11]:
def predict_sentiment(text):

    label = model.predict([text])[0]

    if label == "Negative":
        score = -1
    elif label == "Neutral":
        score = 0
    else:
        score = 1

    return label, score

In [12]:
#Train Topic Model

vectorizer = CountVectorizer(
    stop_words="english",
    max_df=0.9,
    min_df=10
)

X_topics = vectorizer.fit_transform(train_df["text"])

lda_model = LatentDirichletAllocation(
    n_components=6,
    random_state=42
)

lda_model.fit(X_topics)

,n_components,6
,doc_topic_prior,None
,topic_word_prior,None
,learning_method,'batch'
,learning_decay,0.7
,learning_offset,10.0
,max_iter,10
,batch_size,128
,evaluate_every,-1
,total_samples,1000000.0
,perp_tol,0.1


In [13]:
#Topic Predictor

def predict_topic(text):

    vec = vectorizer.transform([text])
    topic_prob = lda_model.transform(vec)
    topic_id = topic_prob.argmax()

    topic_map = {
        0: "Playback",
        1: "Login",
        2: "Ads",
        3: "Other",
        4: "Billing",
        5: "UI/UX"
    }

    return topic_map.get(topic_id, "Other"), float(topic_prob.max())

In [14]:
from datetime import datetime

def convert_reviews(play_reviews, partner_name, partner_record_id):
    
    records = []

    for r in play_reviews:

        review_text = str(r.get("content", "")).strip()
        if not review_text:
            continue

        sentiment_label, sentiment_score = predict_sentiment(review_text)
        topic, confidence = predict_topic(review_text)

        record = {
            "Review ID": r["reviewId"],
            "Review Text": review_text,
            "Rating": r["score"],
            "Date Submitted": r["at"].strftime("%Y-%m-%d"),
            "Partner": [partner_record_id],
            "Source": "GooglePlay",
            "App Version": r.get("appVersion"),
            "Country": "US",
            "Device Type": "Android",
            "Sentiment Label": sentiment_label,
            "Sentiment Score": sentiment_score,
            "Topic": topic,
            "Topic Confidence": confidence,
            "Needs PMO Review": True if sentiment_label == "Negative" else False
        }

        records.append(record)

    return records

In [15]:
import requests

BASE_ID = "YOUR_BASE_ID"
TABLE_NAME = "User Feedback"
API_TOKEN = "Your_Airtable_Token"

headers = {
    "Authorization": f"Bearer {API_TOKEN}",
    "Content-Type": "application/json"
}

def upsert_review(review_data):

    review_id = review_data["Review ID"]
    base_url = f"https://api.airtable.com/v0/{BASE_ID}/{TABLE_NAME}"

    params = {"filterByFormula": f"{{Review ID}}='{review_id}'"}

    r = requests.get(base_url, headers=headers, params=params)
    records = r.json().get("records", [])

    if records:

        record_id = records[0]["id"]
        update_url = f"{base_url}/{record_id}"

        fields_to_update = dict(review_data)
        fields_to_update.pop("Partner", None)

        resp = requests.patch(update_url, headers=headers, json={"fields": fields_to_update})

        print("Updated:", resp.status_code)

    else:

        resp = requests.post(base_url, headers=headers, json={"fields": review_data})

        print("Created:", resp.status_code)

    if resp.status_code not in [200,201]:
        print(resp.text)

In [16]:
disney_reviews, _ = reviews(
    DISNEY_APP_ID,
    lang="en",
    country="us",
    sort=Sort.NEWEST,
    count=200
)

max_reviews, _ = reviews(
    MAX_APP_ID,
    lang="en",
    country="us",
    sort=Sort.NEWEST,
    count=200
)

print("Disney pulled:", len(disney_reviews))
print("Max pulled:", len(max_reviews))

Disney pulled: 200
Max pulled: 200


In [17]:
DISNEY_REC_ID = "reclTLOauWKWXCaQI"
MAX_REC_ID = "recXVtN5llBa2cQU5"

disney_records = convert_reviews(disney_reviews, "Disney+", DISNEY_REC_ID)
max_records = convert_reviews(max_reviews, "Max", MAX_REC_ID)

all_records = disney_records + max_records

print("Records prepared:", len(all_records))

Records prepared: 400


In [42]:
import requests

BASE_ID = "CURRENT BASE ID"   # your current base id
TABLE_NAME = "User Feedback"    # exact Airtable table name
API_TOKEN = "Your_Airtable_API_TOKEN"

url = f"https://api.airtable.com/v0/{BASE_ID}/{TABLE_NAME}"
headers = {"Authorization": f"Bearer {API_TOKEN}"}

resp = requests.get(url, headers=headers)

print("Status:", resp.status_code)
print(resp.text[:500])
print("URL used:", url)

Status: 200
{"records":[{"id":"rec0lB5ppebxfrgPZ","createdTime":"2026-03-14T04:58:28.000Z","fields":{"Review Text":"I am trying to get the Disney+ and Hulu premium bundle and it will not let me so I guess I am just deleting my account?","Rating":1,"Date Submitted":"2026-03-11","Partner":["reclTLOauWKWXCaQI"],"Source":"GooglePlay","App Version":"26.2.1+rc1-2026.02.24","Country":"US","Device Type":"Android","Sentiment Score":-1,"Sentiment Label":"Negative","Topic":"Login","Topic Confidence":0.9239842531205694
URL used: https://api.airtable.com/v0/appM3TaucnFA0YCJ6/User Feedback


In [18]:
import requests

BASE_ID = "appM3TaucnFA0YCJ6"
TABLE_NAME = "User Feedback"
API_TOKEN = "Your_Airtable_API_TOKEN"

headers = {
    "Authorization": f"Bearer {API_TOKEN}",
    "Content-Type": "application/json"
}

def upsert_review(review_data):
    review_id = review_data["Review ID"]
    base_url = f"https://api.airtable.com/v0/{BASE_ID}/{TABLE_NAME}"

    params = {"filterByFormula": f"{{Review ID}}='{review_id}'"}
    r = requests.get(base_url, headers=headers, params=params)

    print("Lookup status:", r.status_code)
    if r.status_code != 200:
        print(r.text)
        return

    records = r.json().get("records", [])

    if records:
        record_id = records[0]["id"]
        update_url = f"{base_url}/{record_id}"

        fields_to_update = dict(review_data)
        fields_to_update.pop("Partner", None)

        resp = requests.patch(update_url, headers=headers, json={"fields": fields_to_update})
        print("Updated:", resp.status_code)
        if resp.status_code != 200:
            print(resp.text)
    else:
        resp = requests.post(base_url, headers=headers, json={"fields": review_data})
        print("Created:", resp.status_code)
        if resp.status_code not in [200, 201]:
            print(resp.text)

In [19]:
import requests

BASE_ID = "appM3TaucnFA0YCJ6"
TABLE_NAME = "User Feedback"
API_TOKEN = "Your_Airtable_API_TOKEN"

url = f"https://api.airtable.com/v0/{BASE_ID}/{TABLE_NAME}"

headers = {
    "Authorization": f"Bearer {API_TOKEN}"
}

resp = requests.get(url, headers=headers)

print("Status:", resp.status_code)
print("Response preview:", resp.text[:300])
print("URL used:", url)

Status: 200
Response preview: {"records":[{"id":"rec0RDkXZlhIixmzV","createdTime":"2026-03-14T05:55:03.000Z","fields":{"Review Text":"cant eveninstal it on my tv it says 100% dounloaded but then it resets me back saying i need to uninstal a app to download and the cycle repeats","Rating":1,"Date Submitted":"2026-03-09","Partner"
URL used: https://api.airtable.com/v0/appM3TaucnFA0YCJ6/User Feedback


In [20]:
import requests

BASE_ID = "appM3TaucnFA0YCJ6"
TABLE_NAME = "User Feedback"
API_TOKEN = "Your_Airtable_API_TOKEN"

headers = {
    "Authorization": f"Bearer {API_TOKEN}",
    "Content-Type": "application/json"
}

def upsert_review(review_data):
    review_id = review_data["Review ID"]
    base_url = f"https://api.airtable.com/v0/{BASE_ID}/{TABLE_NAME}"

    # Step 1: lookup by Review ID
    params = {"filterByFormula": f"{{Review ID}}='{review_id}'"}
    r = requests.get(base_url, headers=headers, params=params)

    print("Lookup status:", r.status_code)

    if r.status_code != 200:
        print("Lookup error:", r.text)
        return

    records = r.json().get("records", [])

    if records:
        # update existing
        record_id = records[0]["id"]
        update_url = f"{base_url}/{record_id}"

        fields_to_update = dict(review_data)
        fields_to_update.pop("Partner", None)

        resp = requests.patch(update_url, headers=headers, json={"fields": fields_to_update})
        print("Updated:", resp.status_code)

        if resp.status_code != 200:
            print("Update error:", resp.text)

    else:
        # create new
        resp = requests.post(base_url, headers=headers, json={"fields": review_data})
        print("Created:", resp.status_code)

        if resp.status_code not in [200, 201]:
            print("Create error:", resp.text)

In [21]:
upsert_review(all_records[0])

Lookup status: 200
Created: 200


In [22]:
import time

for i, r in enumerate(all_records, start=1):
    print(f"Processing {i}/{len(all_records)} | Review ID: {r['Review ID']}")
    upsert_review(r)

Processing 1/400 | Review ID: a35ff659-4fb7-4483-a35c-e4bbd640373d
Lookup status: 200
Updated: 200
Processing 2/400 | Review ID: 2b01b3e1-0956-4ea9-ba39-b4b772f6f9f1
Lookup status: 200
Created: 200
Processing 3/400 | Review ID: 7a66c480-d036-4f13-ab52-23b36e619b5f
Lookup status: 200
Created: 200
Processing 4/400 | Review ID: 8fa56e07-0917-452f-90d6-e4c0b23f363f
Lookup status: 200
Created: 200
Processing 5/400 | Review ID: dcec1554-db26-4e2c-bdbd-580b1862b811
Lookup status: 200
Created: 200
Processing 6/400 | Review ID: 6bd94970-8a68-4b3d-af42-704ff82341b8
Lookup status: 200
Created: 200
Processing 7/400 | Review ID: 8ec744e1-551b-4c9b-a130-e2ca1e67b44f
Lookup status: 200
Created: 200
Processing 8/400 | Review ID: 7032cf1b-c25f-4dca-b577-c231365c9c88
Lookup status: 200
Created: 200
Processing 9/400 | Review ID: 7ff60d3d-1018-4b63-b235-85973a8a742d
Lookup status: 200
Created: 200
Processing 10/400 | Review ID: 29f966fc-ffd2-4e61-97b2-b424c3466ede
Lookup status: 200
Created: 200
Processin